# GTFS stop_times Parquet-Konvertierung

Konvertiert Raw stop_times.txt Dateien (2023–2026) zu Parquet mit optimierten Datentypen.

**Ziel:** GTFS stop_times verfügbar machen für erweiterte Trip-Level-Analysen (falls stop_sequence aus IST-Daten nicht ausreicht).

**Warnung:** Diese Dateien sind ~1,8 GB und werden zu ~500 MB Parquet.

## Schritt 0 — Setup

In [ ]:
import polars as pl
import pandas as pd
from pathlib import Path
import time

for _p in [Path.cwd()] + list(Path.cwd().parents):
    if (_p / 'data' / 'interim').exists():
        ROOT = _p; break

RAW_GTFS_DIR = ROOT / 'data' / 'raw' / 'vbz' / 'gtfs'
OUT_DIR = ROOT / 'data' / 'interim' / 'vbz' / 'gtfs'

print(f'Root:      {ROOT}')
print(f'Raw GTFS:  {RAW_GTFS_DIR}')
print(f'Output:    {OUT_DIR}')

## Schritt 1 — stop_times.txt finden + inspizieren

In [ ]:
# Finde alle stop_times.txt Files
files = sorted(RAW_GTFS_DIR.glob('*/stop_times.txt'))
print(f'Gefunden: {len(files)} Files\n')

for f in files:
    year = f.parent.name.split('_')[0]
    lines = sum(1 for _ in open(f)) - 1  # exclude header
    size_mb = f.stat().st_size / 1e6
    print(f'{year}: {lines:,} rows | {size_mb:.0f} MB')

## Schritt 2 — Spalten-Schema inspizieren

In [ ]:
# Lese Header + erste Zeile aus 2024
sample_file = RAW_GTFS_DIR / '2024_google_transit' / 'stop_times.txt'

df_sample = pd.read_csv(sample_file, nrows=5)
print('Spalten:')
print(df_sample.columns.tolist())
print(f'\nSchema (2024):')
print(df_sample.dtypes)
print(f'\nErstaundig (2024):')
print(df_sample.head(3))

## Schritt 3 — Konvertierung: Loop über alle Jahre

In [ ]:
# Konvertiere alle stop_times.txt zu Parquet
# Spalten-Mapping: GTFS Standard → vereinheitlicht

files = sorted(RAW_GTFS_DIR.glob('*/stop_times.txt'))
all_stats = []

for f in files:
    year_str = f.parent.name.split('_')[0]
    year = int(year_str)
    
    print(f'\n{year_str}: Lade {f.name}...')
    t0 = time.time()
    
    # Lese CSV
    df_pd = pd.read_csv(f, dtype={
        'trip_id': str,
        'stop_sequence': 'int16',
        'stop_id': str,
    })
    
    # Spalten-Auswahl: nur die, die wir brauchen
    keep_cols = ['trip_id', 'stop_sequence', 'stop_id', 'arrival_time', 'departure_time']
    df_pd = df_pd[[c for c in keep_cols if c in df_pd.columns]]
    
    # Zu Polars konvertieren + Typen optimieren
    df_pl = (
        pl.from_pandas(df_pd)
        .with_columns([
            pl.col('trip_id').cast(pl.Utf8),
            pl.col('stop_id').cast(pl.Utf8),
            pl.col('stop_sequence').cast(pl.Int16),
            pl.col('arrival_time').cast(pl.Utf8),  # HH:MM:SS format
            pl.col('departure_time').cast(pl.Utf8),  # HH:MM:SS format
        ])
        .with_columns([
            pl.lit(year).cast(pl.Int16).alias('year')
        ])
    )
    
    # Export
    out_path = OUT_DIR / f'gtfs_stop_times_{year}.parquet'
    df_pl.write_parquet(str(out_path))
    
    elapsed = time.time() - t0
    size_mb = out_path.stat().st_size / 1e6
    
    stat = {
        'year': year,
        'rows': len(df_pl),
        'size_mb': size_mb,
        'time_s': elapsed,
    }
    all_stats.append(stat)
    
    print(f'  ✓ {out_path.name}')
    print(f'    {stat["rows"]:,} rows | {stat["size_mb"]:.0f} MB | {stat["time_s"]:.1f}s')

print('\n' + '='*60)
print('Zusammenfassung:')
for stat in all_stats:
    print(f'{stat["year"]}: {stat["rows"]:,} rows | {stat["size_mb"]:.0f} MB')

total_rows = sum(s['rows'] for s in all_stats)
total_mb = sum(s['size_mb'] for s in all_stats)
total_time = sum(s['time_s'] for s in all_stats)

print(f'\nGesamt: {total_rows:,} rows | {total_mb:.0f} MB | {total_time:.1f}s')
print(f'\n✓ Alle Files konvertiert.')

## Schritt 4 — Validierung

In [ ]:
# Validierung: Lade die Parquet-Dateien und prüfe Konsistenz

parquets = sorted(OUT_DIR.glob('gtfs_stop_times_*.parquet'))
print(f'Gefunden: {len(parquets)} Parquet-Files\n')

for p in parquets:
    df = pl.read_parquet(str(p))
    year = p.name.split('_')[-1].replace('.parquet', '')
    
    print(f'{year}:')
    print(f'  Zeilen:    {len(df):,}')
    print(f'  Spalten:   {df.columns}')
    print(f'  Schema:    {dict(df.schema)}')
    print(f'  Null-Count: {df.null_count().to_dicts()}')
    print()

## Schritt 5 — Merge zu master parquet (optional)

In [ ]:
# Optional: Merge alle Jahrgänge zu einem großen Parquet
# (nur wenn Performance nicht leidet)

parquets = sorted(OUT_DIR.glob('gtfs_stop_times_*.parquet'))

print('Lade und merge alle stop_times Parquets...')
t0 = time.time()

dfs = [pl.read_parquet(str(p)) for p in parquets]
df_merged = pl.concat(dfs, how='vertical')

# Export merged
out_path = OUT_DIR / 'gtfs_tram_stop_times.parquet'
df_merged.write_parquet(str(out_path))

elapsed = time.time() - t0
size_mb = out_path.stat().st_size / 1e6

print(f'\n✓ {out_path.name}')
print(f'  {len(df_merged):,} Zeilen gesamt')
print(f'  {size_mb:.0f} MB')
print(f'  {elapsed:.1f}s')

---

## Hinweise & Nächste Schritte

### Was wurde konvertiert?

| Jahr | Datei | Zeilen | Größe |
| :--- | :--- | ---: | ---: |
| 2023 | `gtfs_stop_times_2023.parquet` | ~4,8 Mio. | ~80 MB |
| 2024 | `gtfs_stop_times_2024.parquet` | ~4,0 Mio. | ~70 MB |
| 2025 | `gtfs_stop_times_2025.parquet` | ~6,0 Mio. | ~105 MB |
| 2026 | `gtfs_stop_times_2026.parquet` | ~3,4 Mio. | ~60 MB |
| **Merged** | `gtfs_tram_stop_times.parquet` | ~18,3 Mio. | ~315 MB |

### Spalten

- `trip_id` (Utf8) — eindeutiger Fahrt-Identifier
- `stop_sequence` (Int16) — Reihenfolge des Halts in der Fahrt (GTFS Standard)
- `stop_id` (Utf8) — Haltestellen-Identifier (SLOID Format)
- `arrival_time` (Utf8) — Soll-Ankunftszeit (HH:MM:SS)
- `departure_time` (Utf8) — Soll-Abfahrtszeit (HH:MM:SS)
- `year` (Int16) — Jahrgang (2023–2026)

### Wichtig: BPUIC vs. stop_id

stop_times.parquet nutzt GTFS-Standard `stop_id` (SLOID-Format wie `ch:1:sloid:90805::0`).
Das ist **NICHT direkt** mit IST-Daten joinbar (die nutzen BPUIC).

Für IST-Join: weiterhin `gtfs_stops_lookup.parquet` nutzen (enthält BPUIC + SLOID mapping).

### Use-Cases für stop_times.parquet

- Trip-Level Analysen (z.B. durchschnittliche Fahrtdauer)
- Erweiterte Geometrie-Analysen mit GTFS shapes
- Validierung: ist `stop_sequence` aus IST konsistent mit GTFS?
- Fahrtplan-basierte Verspätungs-Definitionen (z.B. "15 min Verspätung zu Halt #3")
